# ARC26 OPSD fixed-pool validation

Unsloth/Hugging Face only; no SGLang install and no DFS. The first run is deliberately restricted to one gain-eligible puzzle to verify named-adapter compatibility in the pinned Kaggle environment. The 12 informative keys are pre-split into six development and six untouched confirmation puzzles.

In [ ]:
RUN_EXPERIMENT = False  # Upload safely; set True only with an L4 x4 session.

CODE_DATASET_ROOT = '/kaggle/input/datasets/yuvraj/arc2026'
MODEL_PATH = '/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1'
COMP_ROOT = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2'
FIXED_CANDIDATE_DIR = None  # None auto-discovers an attached pool containing all 32 keys.

PILOT_KEYS = [
    '0934a4d8', '136b0064', '13e47133', '142ca369', '16de56c4', '1818057f',
    '195c6913', '1ae2feb7', '20270e3b', '20a9e565', '21897d95', '247ef758',
    '269e22fb', '28a6681f', '291dc1e1', '2ba387bc', '2c181942', '2d0172a1',
    '31f7f899', '332f06d7', '35ab12c3', '36a08778', '3dc255db', '3e6067c3',
    '409aa875', '4a21e3da', '4c416de3', '5545f144', '581f7754', '58490d8a',
    '58f5dbd5', '5961cc34',
]
INFORMATIVE_KEYS = [
    '142ca369', '1818057f', '1ae2feb7', '20270e3b', '269e22fb', '28a6681f',
    '2ba387bc', '2d0172a1', '36a08778', '3dc255db', '4a21e3da', '58f5dbd5',
]
DEV_KEYS = ['142ca369', '1818057f', '269e22fb', '2d0172a1', '36a08778', '3dc255db']
CONFIRMATION_KEYS = ['1ae2feb7', '20270e3b', '28a6681f', '2ba387bc', '4a21e3da', '58f5dbd5']
SELECTED_KEYS = ['142ca369']  # Smoke; then DEV_KEYS; freeze settings before CONFIRMATION_KEYS.
NPROCS = 1 if len(SELECTED_KEYS) == 1 else 4
END_TIME_HOURS = 0.75 if len(SELECTED_KEYS) == 1 else 3.0

TTFT_METHOD = 'reduced_plus_opsd'
OPSD_COLOR_PERMUTATIONS = 2
OPSD_CROSS_VIEW_PROBABILITY = 0.2
OPSD_MAX_UPDATES = 16
OPSD_LEARNING_RATE = 5e-5
OPSD_TEMPERATURE = 1.0
OPSD_TOP_P = 1.0
OPSD_LAMBDA_CE = 0.0

WORK_ROOT = '/kaggle/working/arc26_opsd_fixed_pool'
WORK_CODE_DIR = WORK_ROOT + '/ARC-AGI1/qwen_baseline'
OUTPUT_DIR = '/kaggle/working/opsd_fixed_pool_scores'
OPSD_LOG_DIR = '/kaggle/working/opsd_logs'
print('RUN_EXPERIMENT =', RUN_EXPERIMENT, 'selected =', SELECTED_KEYS)

In [ ]:
if RUN_EXPERIMENT:
    import os, sys
    os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
    os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
    os.environ['OMP_NUM_THREADS'] = '12'

    import unsloth  # Must precede transformers/peft imports in this pinned stack.
    import numpy as np
    import torch
    import transformers
    import peft
    print('python =', sys.version)
    print('numpy =', np.__version__, np.__file__)
    print('torch =', torch.__version__, torch.__file__)
    print('transformers =', transformers.__version__, transformers.__file__)
    print('peft =', peft.__version__, peft.__file__)
    print('unsloth =', unsloth.__file__)
else:
    print('Dry upload only; no GPU work was started.')

In [ ]:
if RUN_EXPERIMENT:
    import os, shutil, zipfile
    from pathlib import Path

    def candidate_keys(files):
        return {name.split('_', 1)[0] for name in files if '_' in name}

    if FIXED_CANDIDATE_DIR is None:
        discoveries = []
        for root, _dirs, files in os.walk('/kaggle/input'):
            keys = candidate_keys(files)
            if set(PILOT_KEYS).issubset(keys):
                matching = sum(name.split('_', 1)[0] in set(PILOT_KEYS) for name in files)
                discoveries.append((matching, root))
        if discoveries:
            discoveries.sort(reverse=True)
            best_count = discoveries[0][0]
            tied = [root for count, root in discoveries if count == best_count]
            assert len(tied) == 1, {'ambiguous_fixed_candidate_roots': tied}
            FIXED_CANDIDATE_DIR = tied[0]
        else:
            archives = sorted(Path('/kaggle/input').rglob('candidates.zip'))
            assert len(archives) == 1, {'expected_one_candidates_zip': [str(path) for path in archives]}
            extract_root = Path('/kaggle/working/v17_opsd_fixed_candidates_32')
            shutil.rmtree(extract_root, ignore_errors=True)
            extract_root.mkdir(parents=True)
            with zipfile.ZipFile(archives[0]) as archive:
                members = archive.namelist()
                assert len(members) == 647, f'Expected 647 archived candidates, found {len(members)}'
                assert all('/' not in name and name not in {'.', '..'} for name in members), 'Unsafe archive member'
                archive.extractall(extract_root)
            FIXED_CANDIDATE_DIR = str(extract_root)

    fixed_root = Path(FIXED_CANDIDATE_DIR)
    assert fixed_root.is_dir(), fixed_root
    fixed_files = [path for path in fixed_root.iterdir() if path.is_file() and path.name.split('_', 1)[0] in set(PILOT_KEYS)]
    observed_keys = {path.name.split('_', 1)[0] for path in fixed_files}
    assert observed_keys == set(PILOT_KEYS), {'missing': sorted(set(PILOT_KEYS) - observed_keys), 'extra': sorted(observed_keys - set(PILOT_KEYS))}
    assert len(fixed_files) == 647, f'Expected the exact v17 32-puzzle pool (647 files), found {len(fixed_files)}'

    required = [Path(CODE_DATASET_ROOT), Path(MODEL_PATH), Path(COMP_ROOT), Path(os.environ['TRITON_PTXAS_PATH'])]
    for path in required:
        assert path.exists(), path
    print('fixed_candidate_dir =', fixed_root)
    print('verified fixed candidate files =', len(fixed_files))

In [ ]:
if RUN_EXPERIMENT:
    import shutil, zipfile
    from pathlib import Path

    for path in [WORK_ROOT, OUTPUT_DIR, OPSD_LOG_DIR]:
        shutil.rmtree(path, ignore_errors=True)
    for path in Path('/kaggle/working').glob('worker*'):
        if path.is_file():
            path.unlink()

    source_root = Path(CODE_DATASET_ROOT)
    source_candidates = [source_root / 'ARC-AGI1/qwen_baseline', source_root / 'qwen_baseline']
    source_code = next((path for path in source_candidates if path.is_dir()), None)
    assert source_code is not None, source_candidates
    destination = Path(WORK_CODE_DIR)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_code, destination)

    patch_archives = sorted(Path('/kaggle/input').rglob('qwen_baseline_opsd_patch.zip'))
    assert len(patch_archives) == 1, {'expected_one_opsd_patch': [str(path) for path in patch_archives]}
    with zipfile.ZipFile(patch_archives[0]) as archive:
        members = archive.namelist()
        assert all('/' not in name and name not in {'.', '..'} for name in members), 'Unsafe patch archive member'
        archive.extractall(destination)

    for required_name in ['starter.py', 'arc_solver.py', 'arc_opsd.py']:
        assert (destination / required_name).is_file(), destination / required_name
    print('copied base code from', source_code, 'and overlaid', patch_archives[0], 'into', destination)

In [ ]:
if RUN_EXPERIMENT:
    import json, os, subprocess, sys, time

    cmd = [
        sys.executable, 'starter.py',
        '--test-path', COMP_ROOT + '/arc-agi_evaluation_challenges.json',
        '--model-path', MODEL_PATH,
        '--output-dir', OUTPUT_DIR,
        '--keys-json', json.dumps(SELECTED_KEYS),
        '--nprocs', str(NPROCS),
        '--end-time', str(time.time() + END_TIME_HOURS * 3600),
        '--ttft-method', TTFT_METHOD,
        '--fixed-candidate-dir', FIXED_CANDIDATE_DIR,
        '--opsd-log-dir', OPSD_LOG_DIR,
        '--opsd-color-permutations', str(OPSD_COLOR_PERMUTATIONS),
        '--opsd-cross-view-probability', str(OPSD_CROSS_VIEW_PROBABILITY),
        '--opsd-max-updates', str(OPSD_MAX_UPDATES),
        '--opsd-learning-rate', str(OPSD_LEARNING_RATE),
        '--opsd-temperature', str(OPSD_TEMPERATURE),
        '--opsd-top-p', str(OPSD_TOP_P),
        '--opsd-lambda-ce', str(OPSD_LAMBDA_CE),
        '--profile-timings',
    ]
    print('running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=WORK_CODE_DIR, env=os.environ.copy(), check=True)

In [ ]:
if RUN_EXPERIMENT:
    import json, shutil, sys
    from pathlib import Path
    import numpy as np

    if WORK_CODE_DIR not in sys.path:
        sys.path.insert(0, WORK_CODE_DIR)
    from arc_loader import ArcDataset
    from arc_decoder import ArcDecoder, score_kgmon

    def stage_selected(source, destination, keys):
        shutil.rmtree(destination, ignore_errors=True)
        destination.mkdir(parents=True)
        selected = set(keys)
        for path in Path(source).iterdir():
            if path.is_file() and path.name.split('_', 1)[0] in selected:
                shutil.copy2(path, destination / path.name)

    baseline_stage = Path('/kaggle/working/fixed_pool_baseline_selected')
    stage_selected(FIXED_CANDIDATE_DIR, baseline_stage, SELECTED_KEYS)
    data = ArcDataset.from_file(COMP_ROOT + '/arc-agi_evaluation_challenges.json', keys=SELECTED_KEYS)
    data = data.load_replies(COMP_ROOT + '/arc-agi_evaluation_solutions.json')
    split_data = data.split_multi_replies()

    def evaluate(output_dir):
        decoder = ArcDecoder(split_data, n_guesses=2)
        decoder.load_decoded_results(str(output_dir))
        selected = decoder.run_selection_algo(score_kgmon)
        submission = data.get_submission(selected)
        score = data.validate_submission(submission)
        labels = {key: split_data.replies[key][0] for key in split_data.keys}
        top2 = {key: any(np.array_equal(candidate, labels[key]) for candidate in guesses[:2]) for key, guesses in selected.items()}
        oracle = {key: any(np.array_equal(sample['solution'], labels[key]) for sample in decoder.decoded_results[key].values()) for key in decoder.decoded_results}
        return score, top2, oracle

    baseline_score, baseline_top2, baseline_oracle = evaluate(baseline_stage)
    opsd_score, opsd_top2, opsd_oracle = evaluate(Path(OUTPUT_DIR))
    assert baseline_oracle == opsd_oracle, 'Fixed-pool oracle changed; candidate preservation is broken.'
    gains = sorted(key for key in opsd_top2 if opsd_top2[key] and not baseline_top2.get(key, False))
    losses = sorted(key for key in baseline_top2 if baseline_top2[key] and not opsd_top2.get(key, False))
    logs = sorted(Path(OPSD_LOG_DIR).glob('*.json'))
    print('baseline_mean_selector_score =', baseline_score, '/', len(SELECTED_KEYS))
    print('opsd_mean_selector_score =', opsd_score, '/', len(SELECTED_KEYS))
    print('gains =', gains)
    print('losses =', losses)
    print('fixed_pool_oracle_outputs =', sum(baseline_oracle.values()), '/', len(baseline_oracle))
    print('opsd_logs =', [path.name for path in logs])
    for path in logs:
        payload = json.loads(path.read_text())
        stats = payload['stats']
        print(path.stem, 'reserved_C =', payload['reserved_pair_index'], 'accepted =', stats['accepted_updates'], '/', stats['attempted_examples'], 'correction_s =', round(stats['wall_time_s'], 2))